[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/exorbyte/mbox-cookbook/blob/main/06-agentic-ai/07-validating_llm_extracted_data.ipynb)

In [1]:
# !pip install mbox openai python-dotenv

# Validating LLM-Extracted Data

An LLM is genuinely good at pulling structured fields out of a messy support ticket, a customer name, an order number, a product mention. What it hands back is faithful to what was *written*, typos and all, not to what is *true*. Before any of those fields drive an action, updating an order, refunding a customer, they need to be resolved against your actual records, and just as importantly, checked against each other. This notebook works through a case where trusting one resolved field in isolation would have quietly attached the wrong order to the wrong customer.

In this notebook you will:

1. Extract structured fields from a real support ticket with a real LLM call
2. Resolve each extracted field against the tables that actually matter
3. Watch the single highest-scoring resolution turn out to be the wrong one
4. Use a second, independent field to catch it, and see why cross-field consistency is a stronger signal than any one field's score alone

> Note: this notebook makes real calls to the OpenAI API. To run it, put an `OPENAI_API_KEY` in a `.env` file in this directory.

In [2]:
import pandas as pd
from dotenv import load_dotenv
load_dotenv()

True

## 1. The records, and the ticket

Three tables: customers, their orders, and the product catalog from `02`. And one support ticket, written the way customers actually write, a slightly misspelled name, an order number that's easy to fat-finger, a product mentioned in passing rather than by its exact catalog name.

In [3]:
customers = pd.read_csv("datasets/customers.csv")
orders = pd.read_csv("datasets/orders.csv")
catalog = pd.read_csv("datasets/product_catalog.csv")

ticket_text = "Hi, this is Jenifer Alvarez, order ORD-58301. My extendd batery pak arrived but won't hold a charge. Can you help?"
customers

,customer_id,customer_name,email
0,CUST-1001,Jennifer Alvarez,jennifer.alvarez@example.com
1,CUST-1002,Marcus Webb,marcus.webb@example.com
2,CUST-1003,Priya Chandrasekaran,priya.c@example.com
3,CUST-1004,Tom O'Neil,tom.oneil@example.com


## 2. Extract the fields

Ask the model to pull out the customer name, order id, and product mentioned, as a function call rather than free text, so the result is something code can work with directly.

In [4]:
import json
from openai import OpenAI
client = OpenAI()

tools = [{
    "type": "function",
    "function": {
        "name": "extract_ticket_fields",
        "description": "Extract structured fields mentioned in a support ticket.",
        "parameters": {
            "type": "object",
            "properties": {
                "customer_name": {"type": "string"},
                "order_id": {"type": "string"},
                "product_mentioned": {"type": "string"},
            },
            "required": ["customer_name", "order_id", "product_mentioned"]
        }
    }
}]

response = client.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": f"Extract the fields from this support ticket:\n\n{ticket_text}"}],
    tools=tools,
    tool_choice={"type": "function", "function": {"name": "extract_ticket_fields"}}
)
extracted = json.loads(response.choices[0].message.tool_calls[0].function.arguments)
extracted

{'customer_name': 'Jenifer Alvarez',
 'order_id': 'ORD-58301',
 'product_mentioned': 'extendd batery pak'}

The model did its job correctly, `"Jenifer Alvarez"` and `"ORD-58301"` are exactly what the customer typed. Neither is exactly right. That's not an extraction failure, it's the extraction faithfully reporting what's actually in the ticket, resolving it against reality is a separate step, and it's next.

## 3. Resolve each field independently

Three separate M|BOX matches, one per extracted field, each against the table it actually belongs to.

In [5]:
from mbox.indexing import TableIndexer
from mbox.recall import TableRecallConfig, TableRecallFieldConfig, TableRecallMode

def resolve(df, column, query, max_results=3):
    index = TableIndexer.create_index(df, index_columns=[column], tmp_dir=f"tmp_index_{column}")
    config = TableRecallConfig(
        fields=[TableRecallFieldConfig(input_column=column, indexed_column=column,
                                        minimum_quality=0, weight=100, mode=TableRecallMode.APPROX)],
        max_results=max_results, min_total_match_value=0, include_field_scores=True
    )
    return index.match(queries=pd.DataFrame({column: [query]}), config=config)

customer_matches = resolve(customers, "customer_name", extracted["customer_name"])
customer_matches[["customer_name_candidate", "customer_name_score"]]

mbpie: 33 modules, 566 methods, 8 classes, 18 enums
  args: 426 required, 254 optional, 37 keywords, 39 flags, 26 arrays
  types: 372 int, 337 str, 1 double, 72 object


,customer_name_candidate,customer_name_score
0,Jennifer Alvarez,86


A confident, unambiguous resolution: `Jennifer Alvarez`, `CUST-1001`. So far, so good.

In [6]:
order_matches = resolve(orders, "order_id", extracted["order_id"], max_results=3)
order_matches[["order_id_candidate", "order_id_score"]]

,order_id_candidate,order_id_score
0,ORD-58304,78
1,ORD-58291,55
2,ORD-58317,55


The top match is also confident-looking, `ORD-58304` at score 78. If this were the only field being resolved, and the policy from `06` proceeded on any score above, say, 60, this would auto-proceed straight into acting on the wrong order, `ORD-58304` belongs to a different customer entirely.

In [7]:
resolved_customer_id = customers.loc[customers["customer_name"] == customer_matches.iloc[0]["customer_name_candidate"], "customer_id"].iloc[0]
top_order = order_matches.iloc[0]
top_order_owner = orders.loc[orders["order_id"] == top_order["order_id_candidate"], "customer_id"].iloc[0]

print(f"Resolved customer: {resolved_customer_id}")
print(f"Top order match {top_order['order_id_candidate']} (score {top_order['order_id_score']}) belongs to: {top_order_owner}")
print(f"Consistent with resolved customer? {top_order_owner == resolved_customer_id}")

Resolved customer: CUST-1001
Top order match ORD-58304 (score 78) belongs to: CUST-1002
Consistent with resolved customer? False


`False`. The single highest-scoring resolution for `order_id` is not owned by the customer this ticket is actually from. Taken alone, that field's own score gave no hint anything was wrong, `78` reads as a solid match. The problem only becomes visible by checking it against a second, independently resolved fact.

## 4. A second field breaks the tie

The order match ranked second, lower-scoring on its own, is worth a second look precisely because the top one failed the consistency check.

In [8]:
for _, row in order_matches.iterrows():
    owner = orders.loc[orders["order_id"] == row["order_id_candidate"], "customer_id"].iloc[0]
    print(f"{row['order_id_candidate']} (score {row['order_id_score']}) -> owned by {owner}, "
          f"consistent with resolved customer: {owner == resolved_customer_id}")

ORD-58304 (score 78) -> owned by CUST-1002, consistent with resolved customer: False
ORD-58291 (score 55) -> owned by CUST-1001, consistent with resolved customer: True
ORD-58317 (score 55) -> owned by CUST-1003, consistent with resolved customer: False


`ORD-58291`, ranked second on raw text score, is the only candidate actually owned by `CUST-1001`. A third, independent field settles it further: what product did the ticket actually mention?

In [9]:
product_matches = resolve(catalog, "product_name", extracted["product_mentioned"])
resolved_product_id = catalog.loc[catalog["product_name"] == product_matches.iloc[0]["product_name_candidate"], "product_id"].iloc[0]
print(f"Resolved product: {product_matches.iloc[0]['product_name_candidate']} ({resolved_product_id})\n")

for _, row in order_matches.iterrows():
    order_product = orders.loc[orders["order_id"] == row["order_id_candidate"], "product_id"].iloc[0]
    print(f"{row['order_id_candidate']} is for product {order_product}, matches ticket's product: {order_product == resolved_product_id}")

Resolved product: Extended Battery Pack (B88-EXT)

ORD-58304 is for product A12-PWR, matches ticket's product: False
ORD-58291 is for product B88-EXT, matches ticket's product: True
ORD-58317 is for product C99-SNS, matches ticket's product: False


Three independently resolved fields, the customer, the order, and the product, now agree on exactly one record: `ORD-58291`, owned by `CUST-1001`, for product `B88-EXT`, the exact battery pack the ticket describes. That's the record this ticket should update, not the one that happened to score highest on a single fuzzy field.

## 5. The takeaway

No single field's score told the whole story here. `order_id` alone pointed at the wrong customer's order with a score that looked perfectly confident. What caught it was resolving the *other* extracted fields independently and checking whether they agree, not trusting any one resolution in isolation.

**Resolve every extracted field against its own table, don't skip the ones that seem obviously right.** The customer name resolved cleanly and correctly, it would have been easy to assume the order id, extracted from the same trustworthy-looking ticket, deserved the same confidence.

**Cross-check related fields against each other before acting.** A customer and an order, an order and a product, a shipment and an address, wherever your data has a real foreign-key relationship, use it as a consistency check on independently resolved entities, not just as a join you perform after deciding everything is fine.

**Treat a broken consistency check as a signal, not a dead end.** The correct order wasn't discarded here, it was sitting right there in the same result set, just not ranked first by raw text score alone. Combined with `06`, a consistency failure like this is exactly the kind of signal that should route to `ASK_CLARIFYING_QUESTION` or `ESCALATE_TO_HUMAN` when it can't be resolved as cleanly as it was here.

## Next steps

- **`08-bulk_entity_resolution_for_data_agents.ipynb`** - apply this same resolve-and-validate approach across an entire table at once, not one ticket at a time